# Python. Работа с бинарными данными

## Мотивация

Текстовый файл удобен человеку, но миллион чисел в нём занимает много места и
после чтения требует заново восстанавливать форму и типы. Бинарный формат
хранит данные по заранее известным правилам: NumPy восстанавливает массив
вместе с `shape` и `dtype`, HDF5 читает отдельные участки большого набора, а
архив собирает несколько файлов для передачи.

Универсального формата нет. Сначала нужно понять, что именно сохраняется, кто
будет читать файл и нужно ли загружать данные частями. Сегодня разберём пять
вариантов — pickle, NPY, NPZ, HDF5 и TAR — и по дороге повторим NumPy: формы,
оси, типы и маски, без которых ни один из этих файлов не прочитать осмысленно.

> Массивы, срезы и агрегации по осям были на семинаре 1. Здесь они нужны как
> инструмент: важно не «что такое `ndarray`», а сколько он занимает на диске,
> в каком виде уезжает в файл и что вернётся при чтении.

## 0. Подготовка

Понадобятся `numpy` и `h5py`. В готовом окружении курса (учебный сервер, Colab)
они уже стоят; проверить и доставить можно так:

```bash
python3 -c "import numpy, h5py"   # молчит — значит, всё на месте
pip install numpy h5py            # если импорт упал, в активном окружении
```

Если вы собираете окружение сами через `uv` (семинар 5), пакеты добавляют
**внутри проекта**: сначала `uv init`, потом `uv add numpy h5py` — без
`pyproject.toml` команда `uv add` работать не будет.

Все файлы семинара складываем в `~/seminar-14`: `/tmp` вычищается при
перезагрузке машины, а результат нужен и после занятия. Ячейка ниже создаёт
каталог и заводит переменную `WORK`, на которую опираются все дальнейшие
разделы, — ноутбук выполняется сверху вниз, начиная с неё.

In [ ]:
from pathlib import Path

# Рабочий каталог семинара в домашней директории, а не в /tmp:
# /tmp очищается при перезагрузке, а файлы понадобятся на защите.
WORK = Path.home() / "seminar-14"
WORK.mkdir(exist_ok=True)         # если каталог уже есть — ничего не ломаем
print("рабочий каталог:", WORK)

## 1. Pickle: удобно, но только для доверенных данных

`pickle` сохраняет состояние Python-объекта и инструкции, по которым Python
восстановит его при `pickle.load`. Поэтому словарь со списками, массивами и
экземплярами классов можно записать почти без подготовки.

Файл открывают в бинарном режиме: `"wb"` для записи и `"rb"` для чтения.

In [ ]:
import pickle

payload = {"title": "Продажи", "values": [10, 20, 30]}   # обычный словарь
trusted_path = WORK / "trusted.pkl"

with trusted_path.open("wb") as file:      # "wb" — двоичная запись
    pickle.dump(payload, file)             # словарь целиком уехал в файл
with trusted_path.open("rb") as file:      # "rb" — двоичное чтение
    print("прочитали обратно:", pickle.load(file))

### Чем pickle опасен

Главная опасность: загрузка pickle может вызвать указанные в файле функции.
Злоумышленник способен подготовить файл так, что код выполнится уже во время
`pickle.load`. Неизвестный pickle нельзя «просто посмотреть» — опасное действие
начинается при загрузке.

np.load читает данные, pickle.load выполняет инструкции

Метод `__reduce__` может вернуть вызываемый объект и его аргументы — при
загрузке pickle вызовет этот объект. В демонстрации используется безопасный
`print`, чтобы выполнение было видно; злоумышленник указал бы функцию с
опасными действиями.

In [ ]:
class RunOnLoad:
    # __reduce__ отвечает pickle на вопрос «как тебя восстановить».
    # Здесь ответ такой: «вызови print с этой строкой» — и pickle его выполнит.
    def __reduce__(self):
        return (print, ("ОПАСНОСТЬ: функция выполнилась внутри pickle.load()",))

In [ ]:
dangerous_path = WORK / "dangerous.pkl"
with dangerous_path.open("wb") as file:
    pickle.dump({"title": "Продажи", "action": RunOnLoad()}, file)

print("до pickle.load")
with dangerous_path.open("rb") as file:
    loaded = pickle.load(file)     # вот здесь и срабатывает чужая инструкция
print("после pickle.load, loaded['action'] =", loaded["action"])

#### ❓ **Вопрос**: Между строками «до pickle.load» и «после pickle.load» появилось сообщение, хотя программа явно не вызывала `print` в этом месте. Откуда взялся вызов и чему стало равно поле `loaded["action"]`?

<details>

<summary><strong>Ответ</strong></summary>

`RunOnLoad.__reduce__` записал в pickle указание вызвать `print` с заданной строкой. `pickle.load` выполнил этот вызов при восстановлении объекта. `print` возвращает `None`, поэтому поле `loaded["action"]` стало равно `None`. Вместо `print` вредоносный файл может указать опасную функцию.

</details>

Код класса тоже не замораживается внутри файла: pickle хранит имя модуля,
имя класса и состояние объекта. Если класс с тем же именем изменили, загруженный
объект получит текущую реализацию методов. Поэтому pickle годится только для
доверенного внутреннего состояния в контролируемом окружении.

Практический вывод, который пригодится сразу: файл весов `.pt` от `torch.save`
— это zip-контейнер, внутри которого лежит pickle. Выполнится он или нет,
зависит от того, как звать `torch.load`: с `weights_only=False` (и на любом
PyTorch до 2.6, где это было значением по умолчанию) чужие инструкции
исполняются, с `weights_only=True` — нет. Отсюда и предупреждения про
`weights_only`, которые вы увидите в чужом коде. Безопасная замена для весов —
формат `safetensors`: он хранит только тензоры и заголовок и при чтении ничего
не выполняет.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Угроза не теоретическая: на Hugging Face находили выложенные модели,
которые при загрузке открывали злоумышленнику доступ к машине, и площадка
завела автоматическое сканирование pickle-файлов. Отсюда и вся история с
`safetensors`. Заодно стоит проговорить, почему «просто проверить файл перед
загрузкой» не работает: чтобы разобрать pickle, его надо интерпретировать, а
интерпретация и есть опасное действие — безопасный разбор требует отдельного
ограниченного распаковщика, а не `pickle.load`.

</details>

## 2. Массивы: создание, типы, склейка и повторяемость

`ndarray` — массив элементов одного типа. Частые способы создания:

- `np.array(data)` — из списка или другой последовательности;
- `np.arange(start, stop, step)` — значения с заданным шагом;
- `np.linspace(start, stop, num)` — заданное число равномерных точек, включая границы;
- `np.zeros(shape)`, `np.ones(shape)` — массивы из нулей или единиц;
- `np.zeros_like(array)`, `np.ones_like(array)` — та же форма и тип, что у образца.

Главные свойства массива: `shape`, `ndim`, `dtype`, `size`, `nbytes`. Тип задают
при создании через `dtype=` или меняют через `array.astype(np.float32)`.

In [ ]:
import numpy as np

zeros = np.zeros((2, 3), dtype=np.float32)   # две строки, три столбца
ones = np.ones((1, 3), dtype=np.float32)     # одна строка, три столбца
same_shape = np.zeros_like(ones)             # форму и тип берём у образца

print("формы:", zeros.shape, ones.shape, same_shape.shape)
print("тип элементов:", zeros.dtype, "| осей:", zeros.ndim)

`astype` возвращает **новый** массив нужного типа; исходный остаётся
прежним. Тип определяет и то, сколько байт занимает один элемент, — а значит,
и размер будущего файла.

In [ ]:
integers = np.arange(6).reshape(2, 3)   # целые 0..5
grid = np.linspace(0, 1, num=5)         # 5 точек от 0 до 1, границы включены
floats = integers.astype(np.float32)    # копия в вещественном типе

print("grid:", grid)
print("было:", integers.dtype, "-> стало:", floats.dtype)
print("исходный массив не изменился:", integers.dtype)
print("деление целых само даёт:", (integers / 2).dtype)   # дроби включать не надо
print("байт на элемент:", integers.itemsize, "и", floats.itemsize)

`np.concatenate(arrays, axis)` соединяет несколько массивов. У них должно
быть одинаковое число осей, а размеры всех осей, **кроме оси соединения**,
обязаны совпадать. Например, формы `(2, 3)` и `(1, 3)` соединяются по `axis=0`
в `(3, 3)`, а по `axis=1` — не соединяются: там размеры `2` и `1` разные.

In [ ]:
combined = np.concatenate([zeros, ones], axis=0)   # (2, 3) + (1, 3) -> (3, 3)
print("форма:", combined.shape, "| весь массив занимает", combined.nbytes, "байт")

# По axis=1 размеры другой оси (2 и 1) не совпадают — NumPy откажется считать.
try:
    np.concatenate([zeros, ones], axis=1)
except ValueError as error:
    print("ошибка:", error)

Случайные данные должны быть воспроизводимыми, иначе «у меня не
получилось так же» будет звучать на каждом занятии. `np.random.default_rng(seed)`
создаёт **отдельный** генератор со своим состоянием: при одном и том же seed он
выдаёт одну и ту же последовательность. Криптографически безопасными числа от
этого не становятся.

Старый способ — `np.random.seed(seed)` плюс `np.random.normal(...)` — трогает
одно глобальное состояние на всю программу; он ещё встречается в чужом коде, про
него написано в «Дополнительно».

In [ ]:
rng = np.random.default_rng(7)                    # свой генератор, свой seed
first = rng.normal(size=3)
again = np.random.default_rng(7).normal(size=3)   # тот же seed — та же выборка

print("выборка:", first)
print("повторилась при том же seed:", np.array_equal(first, again))

#### ❓ **Вопрос**: Массивы имеют формы `(2, 4)` и `(2, 3)`. По какой оси их удастся соединить, какой получится форма и почему по другой оси не выйдет? И зачем целому массиву может понадобиться `astype(np.float32)` перед делением?

<details>

<summary><strong>Ответ</strong></summary>

Соединятся только по `axis=1`: размер оси 0 у обоих равен `2`, результат — `(2, 7)`. По `axis=0` не выйдет: размеры оси 1 (`4` и `3`) не совпадают, а совпадать обязаны все оси, кроме оси соединения. `astype` нужен не ради самих дробей — деление целых в NumPy и так даёт `float64`. Он задаёт, **какой именно** тип получится: `float32` вдвое меньше на диске и в памяти, и приводить к нему лучше заранее, а не после того, как результат уже занял вдвое больше места. Исходный массив `astype` не меняет.

</details>

## 3. Размерности и оси

`shape` показывает размеры массива по каждой оси. У матрицы формы `(3, 4)` ось
`0` — это три строки, ось `1` — четыре столбца.

`reshape` меняет форму, не меняя количество и **порядок** элементов: 12
последовательных значений можно представить как `(3, 4)`, `(2, 6)` или
`(2, 2, 3)`. `transpose` (он же `.T`) переставляет сами оси, и элементы
начинают читаться в другом порядке. Эти две операции постоянно путают:

reshape против transpose на массиве из 12 элементов

In [ ]:
values = np.arange(12)          # одномерный массив: 0..11
matrix = values.reshape(3, 4)   # те же 12 чисел как 3 строки по 4
print(matrix)
print("форма:", matrix.shape, "| осей:", matrix.ndim, "| элементов:", matrix.size)

In [ ]:
print("reshape(2, 6) — порядок элементов тот же, разбивка другая:")
print(matrix.reshape(2, 6))

print("matrix.T — оси переставлены, форма (3, 4) стала (4, 3):")
print(matrix.T)

Срез у матрицы задают по каждой оси отдельно, через запятую: сначала
строки, потом столбцы. Это понадобится в задачах — двумерные срезы выглядят
непривычно ровно один раз.

In [ ]:
print("одна строка:", matrix[1])              # индекс по первой оси
print("один столбец:", matrix[:, 2])          # ":" — все строки, 2 — номер столбца
print("блок строк 1-2 и столбцов 2-3:")
print(matrix[1:3, 2:4])                       # правая граница не включается
print("строки в обратном порядке:")
print(matrix[::-1])                           # шаг -1 по первой оси

Ось можно и добавить — это нужно, когда функция ждёт матрицу, а у вас
вектор:

- `vector[np.newaxis, :]` превращает вектор формы `(4,)` в строку `(1, 4)`;
- `vector[:, None]` превращает его в столбец `(4, 1)`;
- `np.expand_dims(vector, axis=0)` делает то же самое явной функцией.

In [ ]:
vector = np.arange(4)                        # форма (4,)
row = vector[np.newaxis, :]                  # (1, 4) — одна строка
column = vector[:, None]                     # (4, 1) — один столбец
same_row = np.expand_dims(vector, axis=0)    # то же, что np.newaxis

print("вектор:", vector.shape, "| строка:", row.shape)
print("столбец:", column.shape, "| expand_dims:", same_row.shape)

`np.moveaxis(array, source, destination)` переносит одну выбранную ось.
Классический случай — картинки: библиотеки читают их как `(картинка, высота,
ширина, канал)`, а PyTorch ждёт канал вторым — `(картинка, канал, высота,
ширина)`. Единого стандарта нет: TensorFlow и Keras по умолчанию работают как
раз с каналом в конце, поэтому порядок осей приходится проверять каждый раз.

In [ ]:
images = np.zeros((8, 64, 64, 3), dtype=np.uint8)   # 8 картинок 64x64, 3 канала
channels_first = np.moveaxis(images, -1, 1)         # канал с конца на место 1

print("как читают картинку:", images.shape)
print("как ждёт PyTorch:   ", channels_first.shape)

`flatten()` и `ravel()` вытягивают массив любой формы в одномерный.
Разница в памяти: `flatten()` всегда делает независимую копию, а `ravel()` по
возможности возвращает **вид на ту же память** — и правка результата меняет
исходный массив. Это регулярный источник загадочных багов, поэтому посмотрим
своими глазами.

In [ ]:
copied = matrix.flatten()        # независимая копия
shared = matrix.ravel()          # скорее всего вид на ту же память

shared[0] = 100                  # правим результат ravel...
print("исходная матрица тоже изменилась:", matrix[0, 0])
copied[1] = 200                  # ...а теперь результат flatten
print("а здесь исходник цел:", matrix[0, 1])

ravel даёт вид на ту же память, flatten — копию

#### ❓ **Вопрос**: Почему после правки результата `ravel()` исходный массив иногда меняется, а после правки `flatten()` — никогда? Как это влияет на выбор, когда массив занимает гигабайты?

<details>

<summary><strong>Ответ</strong></summary>

`ravel()` по возможности возвращает вид на ту же память — запись в него меняет исходный массив. `flatten()` всегда копирует, поэтому исходник цел. На больших данных это ещё и вопрос памяти: `flatten()` удваивает потребление, `ravel()` — нет, но требует помнить, что данные общие.

</details>

#### ❓ **Вопрос**: Массив картинок имеет форму `(8, 64, 64, 3)`. Чем отличается результат `np.moveaxis(images, -1, 1)` от `images.reshape(8, 3, 64, 64)`, если форма у них одинаковая?

<details>

<summary><strong>Ответ</strong></summary>

`moveaxis` переставляет оси, и по второй оси действительно оказываются каналы цвета. `reshape` форму даст ту же, но элементы не переставит — он просто нарежет прежнюю последовательность по-новому, и «каналом» вторая ось уже не будет. Одинаковая форма не означает одинаковых данных.

</details>

## 4. Маски и частые вычисления

Арифметика и сравнения применяются сразу ко всему массиву. Выражение
`data > 22` создаёт логическую маску той же формы, а запись `data[data > 22]`
возвращает все подходящие элементы одним массивом.

In [ ]:
data = np.array([
    [10.0, 20.0, 30.0],
    [12.0, 18.0, 33.0],
    [14.0, 22.0, 27.0],
])
print("маска той же формы:\n", data > 22)      # массив из True/False
print("значения по маске:", data[data > 22])   # остались только подходящие

Маской можно выбирать не только отдельные элементы, но и **целые
строки**: одномерная маска длиной в число строк оставляет подходящие строки
целиком. Так отбирают данные по метке или по имени станции — и заметьте, что у
массива строк сравнение `==` тоже поэлементное, в отличие от списка Python.

Условия комбинируют через `&` («и») и `|` («или»), каждое в своих скобках.
`np.nonzero(mask)` по маске возвращает **номера** истинных элементов — пригодится,
когда нужен не сам срез, а индексы.

In [ ]:
stations = np.array(["север", "юг", "север"])   # массив строк, по строке на измерение
print("строки станции «север»:")
print(data[stations == "север"])   # сравнение строк даёт маску, а не одно True

both = (data[:, 0] > 11) & (data[:, 2] < 30)    # два условия сразу
print("номера подходящих строк:", np.nonzero(both)[0])   # nonzero — номера True
print(data[both])

маска той же формы против маски по строкам

Разницу видно по форме результата: маска той же формы, что массив, отдаёт
плоский список значений, а маска длиной в число строк — сами строки целиком.

Сводные значения — `sum`, `mean`, `min`, `max`, `std`, `median`. Параметр
`axis` указывает ось, которая **исчезает** после вычисления: для формы
`(строки, столбцы)` вызов `mean(axis=0)` даёт значение на каждый столбец, а
`mean(axis=1)` — на каждую строку. `argmax` возвращает индекс максимума, а не
само значение.

In [ ]:
print("среднее по столбцам (axis=0):", data.mean(axis=0))
print("std по строкам (axis=1):", data.std(axis=1))
print("медиана по всему массиву:", np.median(data))
print("argmax по строкам:", data.argmax(axis=1))   # это индексы, а не значения

axis указывает ось, которая исчезает после вычисления

Без `axis` у `argmax` тоже есть смысл, но неожиданный: он считает массив
вытянутым в одну строку и возвращает **один** номер. Превратить его обратно в
координаты умеет `np.unravel_index` — эта пара понадобится в задачах M2 и H5.

In [ ]:
print("argmax без axis:", data.argmax())      # номер в «вытянутом» массиве, не координаты
row, column = np.unravel_index(data.argmax(), data.shape)

print("координаты максимума:", row, column)
print("значение там:", data[row, column])

### Отчёт в JSON: numpy-числа туда не кладутся

Каждая задача этого семинара просит сохранить отчёт в JSON — и на этом месте
спотыкаются все. Результаты numpy это не обычные `int` и `float`, а свои типы
(`np.int64`, `np.float32`). Часть из них `json` принимает, часть — нет, и
угадать заранее нельзя: посмотрим на оба случая рядом.

In [ ]:
import json

print("mean прошёл:", json.dumps({"mean": data.mean()}))   # это np.float64
try:
    json.dumps({"max_row": data.argmax()})                 # а это np.int64
except TypeError as error:
    print("argmax не сериализуется:", error)

In [ ]:
# Почему так: np.float64 — подкласс обычного float, а np.int64 подклассом int не является
print("np.float64 — это float?", isinstance(data.mean(), float))
print("np.int64 — это int?  ", isinstance(data.argmax(), int))
print("а np.float32 — float?", isinstance(np.float32(1.5), float))

Лечится приведением к обычным типам Python: `int()` и `float()` для одного
числа, `.tolist()` для массива. Заодно посмотрим на `json.dump` — он пишет
прямо в файл, в отличие от `json.dumps`, которая возвращает строку.

In [ ]:
report = {
    "max_row": int(row),                         # int() — из np.int64 в обычный int
    "column_means": data.mean(axis=0).tolist(),  # tolist() — из массива в список
}
with open(WORK / "report.json", "w", encoding="utf-8") as file:
    json.dump(report, file, indent=2)            # dump пишет в файл, dumps — в строку
print((WORK / "report.json").read_text())

#### ❓ **Вопрос**: Почему `json.dumps` спокойно принял `data.mean()`, но упал на `data.argmax()` — оба ведь получены из numpy?

<details>

<summary><strong>Ответ</strong></summary>

`data.mean()` вернул `np.float64`, а он **подкласс** питоновского `float`, и `json` его сериализует как обычное число. `np.int64` подклассом `int` не является, `np.float32` — не является подклассом `float`, поэтому оба дают `TypeError`. Именно поэтому «в прошлой задаче же работало» — плохой аргумент: пройдёт или нет, зависит от того, какой dtype оказался у результата. Надёжный способ один — приводить явно.

</details>

`np.matmul(left, right)` и оператор `left @ right` — матричное умножение:
внутренние размеры должны совпасть, внешние остаются.

In [ ]:
weights = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])
product = data @ weights          # (3, 3) @ (3, 2) -> (3, 2)
print("форма результата:", product.shape)
print("@ и np.matmul — одно и то же:", np.array_equal(product, np.matmul(data, weights)))

#### ❓ **Вопрос**: `data.argmax(axis=1)` вернул `[2, 2, 2]`. Означает ли это, что максимумы в строках равны двум? Какой формы будет результат `data @ weights` для форм `(3, 3)` и `(3, 2)`?

<details>

<summary><strong>Ответ</strong></summary>

Нет, числа `2` — индексы столбцов, где находятся максимумы. Матричное произведение имеет форму `(3, 2)`: внешние размеры остаются, внутренние размеры `3` должны совпасть.

</details>

### Пропуски: как `null` из JSON становится `NaN`

Реальные данные приходят дырявыми: в JSON на месте несделанного измерения стоит
`null`. NumPy умеет принять его сам — если попросить вещественный тип, `None`
превращается в `NaN`, «не число». С этого шага начинаются задачи H5 и H6.

In [ ]:
rows = [[1.0, 10.0, None], [2.0, None, 51.0], [3.0, 12.0, 49.0]]   # так выглядит JSON
# asarray = array, но без копирования, если вход уже массив нужного типа
table = np.asarray(rows, dtype=np.float64)   # None становится NaN, потому что тип float

print(table)
print("обычное среднее по столбцам:", table.mean(axis=0))      # NaN заражает результат
print("среднее без пропусков:", np.nanmean(table, axis=0))     # nan-версия их игнорирует

`np.where(condition)` возвращает координаты истинных элементов — два
массива: номера строк и номера столбцов. Парой таких массивов можно сразу и
читать, и **записывать**: `filled[gap_rows, gap_columns] = ...` меняет ровно
найденные ячейки.

In [ ]:
means = np.nanmean(table, axis=0)
gap_rows, gap_columns = np.where(np.isnan(table))    # где именно дырки

filled = table.copy()                                # исходник не трогаем
filled[gap_rows, gap_columns] = means[gap_columns]   # каждой дырке — среднее её столбца
print(filled)
print("пропусков не осталось:", np.isfinite(filled).all())

#### ❓ **Вопрос**: Почему `table.mean(axis=0)` вернул `nan` в тех столбцах, где был `null`, а `np.nanmean` — обычное число? И почему пропуски нельзя просто заменить нулями?

<details>

<summary><strong>Ответ</strong></summary>

`NaN` означает «неизвестно», и любая арифметика с ним даёт `NaN` — обычное среднее «заражается» одним пропуском. `np.nanmean` считает по тем значениям, которые есть. Нули вместо пропусков — это не «нет данных», а «измерили ноль»: они сдвинут среднее и стандартное отклонение, то есть подменят данные выдумкой.

</details>

### Z-нормализация: столбцы в одном масштабе

У давления сотни единиц, у температуры — десятки. Пока столбцы в разных
масштабах, их нельзя ни сравнивать между собой, ни подавать в модель: большие
числа перевесят маленькие просто потому, что они большие.

Лечит это **z-нормализация**: из каждого столбца вычитают его среднее и делят на
его стандартное отклонение, `(x - mean) / std`. После неё у каждого столбца
среднее равно нулю, а разброс — единице. Вычитание массива формы `(столбцы,)`
из матрицы NumPy сделает сам, по broadcasting: каждая строка обработается
одинаково. Этот приём нужен в задачах M1, M4, M6, H3 и H5.

In [ ]:
mean_by_column = filled.mean(axis=0)     # по одному числу на столбец
std_by_column = filled.std(axis=0)

normalized = (filled - mean_by_column) / std_by_column   # broadcasting по столбцам
print(normalized.round(3))
print("среднее после:", normalized.mean(axis=0).round(9))
print("разброс после:", normalized.std(axis=0))

Одна засада: если столбец постоянный, его стандартное отклонение равно
нулю и деление даст `nan`. Постоянный столбец не несёт информации, но ронять
из-за него весь конвейер незачем — единицу вместо нуля подставляют через
`np.where`.

In [ ]:
constant = np.array([[1.0, 5.0], [2.0, 5.0], [3.0, 5.0]])   # второй столбец постоянный
column_std = constant.std(axis=0)
print("стандартные отклонения:", column_std)                # у второго столбца 0

safe_std = np.where(column_std == 0, 1, column_std)         # на ноль не делим
print((constant - constant.mean(axis=0)) / safe_std)

## 5. NPY и NPZ

NPY хранит **один** массив вместе с его `shape` и `dtype`: `np.save(path, array)`
записывает, `np.load(path)` восстанавливает — не нужно ни парсить текст, ни
помнить форму.

У `np.load` параметр `allow_pickle` по умолчанию равен `False`. Это запрещает
автоматическую загрузку Python-объектов из NPY/NPZ и защищает от той же
опасности, что у pickle. Для числовых массивов оставляем `False`.

In [ ]:
sensors = np.arange(300, dtype=np.float32).reshape(100, 3)  # 100 измерений, 3 канала
sensors_path = WORK / "sensors.npy"

np.save(sensors_path, sensors)                          # форма и тип уедут в файл
restored = np.load(sensors_path, allow_pickle=False)    # объекты Python запрещены

print("форма и тип восстановились сами:", restored.shape, restored.dtype)
print("значения совпали:", np.array_equal(sensors, restored))

Мотивация обещала, что бинарный формат экономит место. Проверим на тех же
числах: сохраним их ещё и текстом, как сохранил бы `savetxt` или CSV-экспорт.

In [ ]:
text_path = WORK / "sensors.txt"
np.savetxt(text_path, sensors)                    # те же 300 чисел, но текстом

text_size = text_path.stat().st_size
binary_size = sensors_path.stat().st_size
print("текст:", text_size, "байт | npy:", binary_size, "байт")
print("во сколько раз больше текст:", round(text_size / binary_size, 1))

Внутри NPY нет ничего волшебного: несколько служебных байт, а дальше
сами числа. Заголовок специально сделан читаемым — посмотрим на него.

In [ ]:
print(sensors_path.read_bytes()[:64])   # \x93NUMPY, версия и словарь с shape и dtype

`mmap_mode="r"` отображает NPY в память и подгружает только те участки, к
которым вы обратились. Это спасает, когда файл больше доступной оперативной
памяти, а нужны из него отдельные строки. Режим `"r"` не разрешает менять файл.

обычная загрузка против mmap_mode

In [ ]:
mapped = np.load(sensors_path, mmap_mode="r", allow_pickle=False)

print("тип объекта:", type(mapped).__name__)     # memmap, а не обычный ndarray
print("три строки из середины:", mapped[40:43])  # в память попали только они

NPZ хранит **несколько именованных** массивов в одном файле. `np.savez`
только упаковывает, `np.savez_compressed` ещё и сжимает: сжатие уменьшает размер
не для любых данных и стоит процессорного времени.

Важное отличие от NPY: из NPZ массив достают **по имени и целиком**. Трюк с
`mmap_mode` здесь не работает — выбранный массив сначала распаковывается в
память, и только потом от него можно взять срез.

In [ ]:
labels = np.arange(100, dtype=np.int16) % 2      # по метке на каждое измерение
plain_path = WORK / "dataset-plain.npz"
dataset_path = WORK / "dataset.npz"

np.savez(plain_path, sensors=sensors, labels=labels)             # только упаковать
np.savez_compressed(dataset_path, sensors=sensors, labels=labels)  # ещё и сжать
print("npy:", sensors_path.stat().st_size, "| npz:", plain_path.stat().st_size,
      "| сжатый npz:", dataset_path.stat().st_size)

Несжатый NPZ больше NPY по понятной причине: внутри не только измерения,
но и массив меток, и оболочка zip на каждого участника. А вот сжатие здесь
уменьшило файл в два раза — но только потому, что данные искусственно ровные
(`arange` и чередующиеся метки). На реальных измерениях выигрыш куда скромнее,
а на уже сжатых данных вроде JPEG его не будет вовсе.

In [ ]:
with np.load(dataset_path, allow_pickle=False) as archive:   # закроется сам
    print("имена массивов внутри:", archive.files)
    print("массив достаётся по имени, целиком:", archive["sensors"].shape)

#### ❓ **Вопрос**: Файл NPY занимает 20 ГБ, но программе нужны строки с 1000-й по 1999-ю. Как открыть его без загрузки всех 20 ГБ? Почему совет «поставить `allow_pickle=True`, если файл не читается» опасен?

<details>

<summary><strong>Ответ</strong></summary>

Используют `np.load(path, mmap_mode="r", allow_pickle=False)` и затем берут нужный срез. `allow_pickle=True` разрешает восстановление Python-объектов с выполнением инструкций pickle; неизвестному файлу нельзя давать такое разрешение.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

То же самое видно и из терминала: `head -c 80 ~/seminar-14/sensors.npy`.
Полезная привычка — когда файл «не читается», сначала посмотреть на его первые
байты, а не искать магический флаг в документации. По первым байтам узнаются и
остальные форматы: `\x1f\x8b` — gzip, `PK` — zip (а значит, и `.docx`, и
`.pt` от torch), `\x89HDF` — HDF5. Отсюда же берётся команда `file`, которая
угадывает тип по такой «сигнатуре», а не по расширению.

</details>

## 6. HDF5: большое дерево данных в одном файле

HDF5 — один файл с внутренним деревом:

- **group** похожа на каталог и объединяет связанные данные;
- **dataset** похож на массив NumPy, но остаётся на диске;
- **attribute** хранит небольшое описание: единицы, версию, источник.

дерево групп, датасетов и атрибутов внутри одного .h5

Путь `experiment/values` означает dataset `values` внутри группы `experiment`.
У dataset есть `shape`, `dtype` и срезы: `dataset[100:200]` поднимает с диска
только нужный кусок, а `dataset[:]` загружает dataset целиком.

Режим `"w"` создаёт файл заново, `"r"` открывает для чтения, `"a"` сохраняет
существующее содержимое и разрешает добавление. Блок `with` закрывает файл даже
при ошибке. Это формат файла, а не Hadoop HDFS — созвучие обманчиво.

Положим в HDF5 **те же измерения**, которые только что сохраняли в NPY. Заодно
попросим хранить dataset блоками по 20 строк (`chunks`) и сжимать каждый блок
(`compression`): именно из-за этого HDF5 и берут для больших наборов — читается
и распаковывается только нужный блок, а не весь массив.

Компрессия здесь выбрана `"gzip"`, потому что она есть в любой сборке HDF5 —
файл прочитают и чужой программой, и на другой машине. Альтернативы (`lzf`
быстрее, но сжимает слабее; сторонние фильтры сжимают сильнее, но требуют
плагина у читателя) — в «Дополнительно».

Форму блока подбирают под типичный запрос: читаете по сотне соседних строк —
блок в сотню строк и берите. Слишком большой блок означает распаковку лишнего
на каждое чтение, слишком мелкий — много служебных обращений к диску. У нас
данных всего 100 строк, поэтому блок в 20 строк — просто «пятая часть набора».

In [ ]:
import h5py

measurements_path = WORK / "measurements.h5"

with h5py.File(measurements_path, "w") as file:      # "w" — создать заново
    group = file.create_group("experiment")          # группа = «каталог» внутри файла
    # chunks — блок, которым dataset лежит на диске; сжимается каждый блок отдельно
    group.create_dataset("values", data=sensors, chunks=(20, 3), compression="gzip")

print("h5:", measurements_path.stat().st_size, "| npy:", sensors_path.stat().st_size)

Обратите внимание: на этих ста строках HDF5 получился **больше** NPY.
Так и должно быть — дерево, оглавление и описания блоков занимают место, и на
крошечном наборе служебная часть перевешивает данные. Выигрыш начинается там,
где массив не влезает в память целиком, а читать нужно кусками.

Данные без описания через полгода никто не расшифрует, поэтому рядом с
ними кладут атрибуты. Открываем файл в режиме `"a"` — он дописывает, не стирая
уже записанное.

In [ ]:
with h5py.File(measurements_path, "a") as file:              # "a" — дописать
    file["experiment/values"].attrs["units"] = "mV"          # подпись к dataset
    file.attrs["description"] = "Учебные измерения"          # подпись ко всему файлу
    print("атрибуты файла:", dict(file.attrs))               # видно, что записалось

In [ ]:
with h5py.File(measurements_path, "r") as file:      # "r" — только чтение
    dataset = file["experiment/values"]              # путь как в файловой системе
    print("путь внутри файла:", dataset.name)
    print("форма и тип:", dataset.shape, dataset.dtype)
    print("блок и сжатие:", dataset.chunks, dataset.compression)
    print("единицы измерения:", dataset.attrs["units"])
    print("три строки с диска:", dataset[10:13])     # поднялся только нужный чанк

Чужой файл вы получаете без карты: путь `experiment/values` заранее
неизвестен. `visititems` обходит всё дерево и показывает, что внутри, — с этого
начинают знакомство с любым `.h5`.

In [ ]:
with h5py.File(measurements_path, "r") as file:
    print("верхний уровень:", list(file.keys()))         # что лежит в корне
    # visititems зовёт функцию для каждого узла: путь и сам объект
    file.visititems(lambda name, node: print(" ", name, type(node).__name__))
    print("подписи файла:", dict(file.attrs))

#### ❓ **Вопрос**: Dataset из миллиона строк лежит блоками `chunks=(20, 3)`. Сколько строк поднимется с диска при `dataset[10:13]`, сколько — при `dataset[:][10:13]`, и что изменится, если сделать `chunks=(100000, 3)`?

<details>

<summary><strong>Ответ</strong></summary>

`dataset[10:13]` прочитает и распакует один блок из 20 строк — не три, но и не миллион. `dataset[:]` сначала поднимет в память **весь** миллион строк и только потом отдаст срез. С блоком в 100000 строк каждое такое чтение распаковывало бы по сто тысяч строк ради трёх: форму блока подбирают под то, как данные читают.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

HDF5 придумали не в индустрии, а в научных центрах: формат вырос в NCSA
для суперкомпьютерных расчётов, и сегодня в нём лежат данные климатических
моделей, спутниковой съёмки и физических установок. Отсюда и характерная черта:
файл самоописывающий, через десять лет его можно открыть и понять, что внутри,
не имея исходного кода — ровно то, чего не даёт pickle.

</details>

## 7. TAR и компрессия

TAR собирает несколько файлов и каталогов в один поток, но сам по себе не
сжимает:

- `"w"` создаёт обычный `.tar`;
- `"w:gz"` создаёт `.tar.gz` с gzip;
- `"w:xz"` создаёт `.tar.xz` с более медленной, но часто более сильной компрессией;
- `"r:*"` при чтении определяет вариант автоматически.

tar.gz сжимает поток целиком, zip — каждый файл отдельно

In [ ]:
import shutil

source = WORK / "bundle"
source.mkdir(exist_ok=True)
shutil.copy(sensors_path, source / "sensors.npy")     # те же измерения из раздела 5
(source / "meta.json").write_text('{"rows": 100, "columns": 3}\n', encoding="utf-8")
(source / "readme.txt").write_text("измерения датчика\n" * 100, encoding="utf-8")

print("что кладём в архив:", sorted(path.name for path in source.iterdir()))

`arcname` задаёт **внутреннее** имя участника — иначе в архив уедет
абсолютный путь с компьютера автора, вместе с именем пользователя.

In [ ]:
import tarfile

simple_path = WORK / "bundle.tar.gz"
with tarfile.open(simple_path, "w:gz") as archive:   # "w:gz" = tar + gzip
    archive.add(source, arcname="bundle")            # внутри будет bundle/...

print("три файла на диске:", sum(p.stat().st_size for p in source.iterdir()), "байт")
print("весь архив:", simple_path.stat().st_size, "байт")   # и текст, и числа сжались

`extractfile` открывает одного участника как файловый объект — распаковывать
весь архив на диск не нужно, такой объект можно сразу отдать в `json.load`.

In [ ]:
import json

with tarfile.open(simple_path, "r:*") as archive:    # "r:*" — сжатие определит само
    print("участники:", [member.name for member in archive.getmembers()])
    with archive.extractfile("bundle/meta.json") as file:
        metadata = json.load(file)                   # читаем прямо из архива

print("прочитали, ничего не распаковывая:", metadata)

А вот с NPY тот же приём внезапно не работает. Посмотрим, как именно он
ломается, — эта ошибка встретится в задачах M5 и M6.

In [ ]:
with tarfile.open(simple_path, "r:*") as archive:
    with archive.extractfile("bundle/sensors.npy") as file:
        try:
            np.load(file, allow_pickle=False)   # выглядит как обычный открытый файл
        except AttributeError as error:
            print("не вышло:", error)

Причина в том, что участник архива только **притворяется** файлом:
`extractfile` возвращает объект-наследник `io.BufferedReader`, NumPy принимает
его за настоящий файл и просит у него дескриптор (`fileno`) ради быстрого
чтения. Дескриптора нет: отдельного файла на диске у участника не существует,
его байты лежат внутри общего `.tar.gz`.

Лечится это тем, что байты участника читают в `io.BytesIO` — «файл в памяти»,
который ведёт себя предсказуемо.

In [ ]:
import io

with tarfile.open(simple_path, "r:*") as archive:
    with archive.extractfile("bundle/sensors.npy") as file:
        buffer = io.BytesIO(file.read())                    # байты участника — в память
        from_archive = np.load(buffer, allow_pickle=False)  # с буфером всё получается

print("массив из архива:", from_archive.shape, np.array_equal(from_archive, sensors))

### Почему одинаковые архивы получаются разными

Контрольная сумма архива — самый дешёвый способ доказать, что две команды учили
модель на одних и тех же данных, а пакет собран из тех самых исходников. Работает
это только если одинаковые данные дают одинаковые байты, а по умолчанию — не дают.

Архив сохраняет изменяемые метаданные файлов: время изменения, права и
владельца. Заголовок gzip тоже содержит время создания и имя. Поэтому два архива
с одинаковым содержимым легко отличаются байтами — и контрольные суммы не
совпадают, хотя данные те же.

Проверим это на опыте.

In [ ]:
import hashlib
import os

def digest(path):                    # контрольная сумма файла
    return hashlib.sha256(path.read_bytes()).hexdigest()

def set_time(seconds):               # переставить время изменения у исходных файлов
    for path in sorted(source.iterdir()):
        os.utime(path, (seconds, seconds))

In [ ]:
def pack_simple(path):                       # обычная сборка
    with tarfile.open(path, "w:gz") as archive:
        archive.add(source, arcname="bundle")

Собираем дважды в **один и тот же файл** — иначе разошлись бы ещё и имена
в заголовке gzip. Между сборками меняем у исходных файлов время изменения.

Строго говоря, источников различий здесь два: время файлов, которое мы меняем
сами, и текущее время, которое `"w:gz"` кладёт в заголовок gzip — если сборки
попали в разные секунды, разойдётся и оно. Детерминированный вариант ниже
закрывает оба.

In [ ]:
simple = WORK / "simple.tar.gz"
set_time(1_000_000)
pack_simple(simple)
first_sum = digest(simple)

set_time(2_000_000)                          # содержимое то же, время другое
pack_simple(simple)
print("суммы совпали:", first_sum == digest(simple))

Чтобы архив стал детерминированным, нужно убрать всё изменчивое:

1. добавлять файлы в отсортированном порядке;
2. занулить `mtime`, `uid`, `gid`, `uname`, `gname` и зафиксировать `mode` у каждого участника TAR;
3. поставить `mtime=0` и пустое имя в заголовке gzip.

Пункт 2 удобно сделать фильтром: `tarfile` пропускает через него заголовок
каждого участника и берёт то, что функция вернёт.

In [ ]:
def freeze(info):          # tarfile зовёт эту функцию для каждого участника
    info.mtime = 0         # время изменения
    info.uid = 0           # владелец
    info.gid = 0           # группа
    info.uname = ""        # имя владельца текстом
    info.gname = ""        # имя группы текстом
    info.mode = 0o644      # права
    return info

In [ ]:
import gzip

def pack_frozen(path):
    with path.open("wb") as output:
        # filename="" и mtime=0 — имя и время не попадут в заголовок gzip
        with gzip.GzipFile(filename="", mode="wb", fileobj=output, mtime=0) as packed:
            with tarfile.open(fileobj=packed, mode="w") as archive:
                for item in sorted(source.iterdir()):    # порядок фиксирован
                    archive.add(item, arcname=f"bundle/{item.name}", filter=freeze)

In [ ]:
frozen = WORK / "frozen.tar.gz"
set_time(1_000_000)
pack_frozen(frozen)
first_sum = digest(frozen)
set_time(2_000_000)                          # снова меняем время файлов
pack_frozen(frozen)
print("суммы совпали:", first_sum == digest(frozen))

В заголовок gzip действительно попадает имя: `tarfile.open(..., "w:gz")`
берёт его от имени файла. Посмотрим на первые байты обоих архивов — после
сигнатуры `\x1f\x8b` идут флаги, время и, если оно записано, имя.

In [ ]:
print("обычный:", simple.read_bytes()[:24])
# у детерминированного на месте времени нули, а имени нет вовсе
print("замороженный:", frozen.read_bytes()[:24])

#### ❓ **Вопрос**: Почему два обычных `.tar.gz` с одинаковыми файлами могут иметь разные контрольные суммы? Что фиксирует детерминированный вариант?

<details>

<summary><strong>Ответ</strong></summary>

В заголовки TAR и gzip попадают изменяемые метаданные, прежде всего время. Детерминированный вариант сортирует участников и фиксирует время, владельца, группу, права и данные заголовка gzip. При одинаковом содержимом архивы после этого совпадают побайтно — это и показали две пары контрольных сумм.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Зачем вообще побайтно одинаковые архивы — это движение reproducible
builds. Если сборка воспроизводима, любой может собрать пакет из тех же
исходников и убедиться, что опубликованный бинарник получен именно из них, а не
из подменённого кода. У GNU tar для этого есть `--sort=name --mtime=...
--owner=0 --group=0`, а сборочные системы договорились о переменной окружения
`SOURCE_DATE_EPOCH`. В ML та же идея работает для датасетов: одинаковая
контрольная сумма архива — самый дешёвый способ доказать, что две команды учили
модель на одних и тех же данных.

</details>

Компрессия хорошо работает на повторяющемся тексте и массивах с
закономерностями. Уже сжатые JPEG, MP4 или NPZ почти не уменьшаются, а очень
маленький архив иногда даже становится больше из-за служебных данных. Проверка
чужих архивов перед извлечением вынесена в «Дополнительно» — извлекать
неизвестный архив «как есть» нельзя.

## 8. Что выбирать

| Формат | Что хранит | Чтение части данных | Встроенная компрессия | Типичный выбор |
|---|---|---:|---:|---|
| NPY | один NumPy-массив с формой и типом | да, через `mmap_mode` | нет | промежуточный числовой массив |
| NPZ | несколько именованных массивов | только массив целиком, по имени | по выбору | небольшой набор массивов |
| HDF5 | дерево групп, datasets и атрибутов | да | по dataset | большой структурированный набор |
| pickle | почти любой объект Python | нет | нет | только доверенное внутреннее состояние |
| TAR | несколько готовых файлов | по участникам | через gzip/xz | доставка каталога или результата |

#### ❓ **Вопрос**: Таблица измерений занимает 50 ГБ, программа читает из неё то один диапазон строк, то другой, и рядом надо хранить описание: единицы, дату, версию прибора. Что выбрать и почему не подойдут NPZ и pickle? А если этот же файл нужно отдать коллеге вместе с README и отчётом?

<details>

<summary><strong>Ответ</strong></summary>

Для самих данных — HDF5: он читает выбранные строки с диска и хранит описание в атрибутах рядом с dataset. NPZ пришлось бы распаковывать целиком на каждое чтение, а pickle к тому же выполняет инструкции из файла и не даёт частичного чтения. Подошёл бы и NPY с `mmap_mode`, но описание пришлось бы класть отдельным файлом. Для передачи комплекта — TAR или TAR.GZ: он не заменяет формат данных, а собирает готовые файлы в один.

</details>

## Дополнительно

### Старый способ: `np.random.seed`

В коде старше нескольких лет вместо генератора встречается пара

```python
np.random.seed(42)
sample = np.random.normal(size=5)
```

`np.random.seed` задаёт состояние **одного глобального** генератора, общего для
всей программы. Любая библиотека, которая по дороге дёрнет `np.random.*`,
сдвинет последовательность, и воспроизводимость сломается незаметно. Читать
такой код нужно уметь, писать новый — через `np.random.default_rng(seed)`.

### `io.BytesIO`: чего он стоит

Приём из раздела 7 в том виде, в каком он нужен задачам M5 и M6 (файл берётся
из `assets/M5/dataset.tar.gz` после `./generate.sh`):

```python
import io
import tarfile
import numpy as np

with tarfile.open("assets/M5/dataset.tar.gz", "r:*") as archive:
    with archive.extractfile("dataset/features.npy") as file:
        buffer = io.BytesIO(file.read())
        features = np.load(buffer, allow_pickle=False)
```

Так распаковывается только выбранный участник, а не весь TAR. Но сам выбранный
NPY при этом целиком попадает в RAM — `mmap_mode` поверх `BytesIO` смысла не
имеет. Для очень большого массива лучше хранить NPY отдельным файлом или читать
HDF5 по срезам.

### Поиск, выбор и сортировка

Частые функции, которые полезно узнавать в рабочем коде:

- `np.where(condition, yes, no)` выбирает значение для каждого элемента;
- `np.where(condition)` и `np.nonzero(condition)` возвращают координаты истинных элементов;
- `np.sort(array, axis=...)` возвращает отсортированную копию;
- `np.argsort(array)` возвращает индексы, задающие порядок;
- `np.unravel_index(flat_index, shape)` переводит один индекс в координаты многомерного массива;
- `np.ptp(array)` вычисляет размах `max - min`;
- `np.unique(array, return_counts=True)` находит уникальные значения и частоты;
- `np.clip(array, low, high)` ограничивает значения диапазоном;
- `np.percentile(array, [25, 50, 75])` считает квантили.

`array.nonzero()` — метод массива; имя функции NumPy пишется слитно:
`np.nonzero`, не `non_zero`.

Чтобы список не остался мёртвым перечнем, вот он в работе:

In [ ]:
noisy = np.array([5.0, -3.0, 12.0, 5.0, 40.0])

print("сортировка:", np.sort(noisy), "| порядок индексов:", np.argsort(noisy))
print("уникальные и счётчики:", np.unique(noisy, return_counts=True))
print("размах:", np.ptp(noisy), "| квартили:", np.percentile(noisy, [25, 50, 75]))
print("обрезали по 10:", np.clip(noisy, 0, 10))
print("выбор по условию:", np.where(noisy > 5, "много", "мало"))

### HDF5: chunks, компрессия и рост dataset

HDF5 хранит chunked dataset блоками. Форма chunk должна соответствовать
типичному чтению: если программа постоянно берёт сотни соседних строк, chunk на
несколько сотен строк обычно разумнее одной огромной плитки.

```python
rows = min(1000, values.shape[0])   # chunk не может быть больше самого dataset
dataset = file.create_dataset(
    "values",
    data=values,
    chunks=(rows, values.shape[1]),
    compression="gzip",
    compression_opts=4,
    shuffle=True,
)
```

Ограничение важное: если попросить `chunks=(1000, 3)` у dataset из 20 строк,
h5py откажется — `Chunk shape must not be greater than data shape`.

`gzip` переносим и обычно хорошо сжимает; `lzf` в h5py работает быстрее, но
часто сжимает слабее. Компрессия применяется к chunks, поэтому чтение одного
элемента всё равно может потребовать чтения и распаковки целого chunk.

Dataset можно создать растущим через `maxshape=(None, columns)` и увеличить
методом `resize`. Структуру незнакомого файла смотрят через `list(file.keys())`,
`dataset.name`, `shape`, `dtype`, `attrs` или обход `file.visititems(...)`. Для
больших данных не вызывают `dataset[:]` без необходимости.

### Пропуски и нормализация: что ещё проверить

Пропуски, `np.nanmean`, заполнение по координатам и z-нормализацию с `safe_std`
разобрали в разделе 4. Что стоит добавить в собственном конвейере:

- результат проверяют вызовом `np.isfinite(normalized).all()` — он ловит и
  оставшиеся `NaN`, и бесконечности от деления на почти-ноль;
- заполнять пропуски средним — не единственный вариант: медиана устойчивее к
  выбросам, а иногда правильнее выкинуть строку целиком;
- сами `mean` и `std` сохраняют рядом с результатом: без них новые данные не
  привести к тому же масштабу.

### Проверяемый конвейер

Сначала проверяют `shape`, `ndim` и числовой `dtype` исходного массива. Затем
создают результат, не меняя исходный файл, сохраняют его и параметры
преобразования, повторно открывают созданный файл и проверяют форму, тип и
значения.

Средние, стандартные отклонения и другие параметры сохраняют рядом с
результатом: они нужны для воспроизводимости и обработки новых данных по тем же
правилам.

### Миграция pickle

Доверенный pickle можно загрузить один раз и перевести в более узкие форматы.
NumPy-массивы сохраняют через `np.save` или HDF5, JSON-совместимые метаданные —
через `json.dump`. После переноса новые файлы читают обратно и сравнивают с
исходным объектом. Недоверенный pickle нельзя сделать безопасным
«предварительной конвертацией»: конвертер всё равно сначала выполнит
`pickle.load`.

### Безопасное извлечение TAR

До первого извлечения проверяют **всех** участников. `PurePosixPath(member.name)`
помогает обнаружить абсолютный путь и компонент `..`. `member.issym()` и
`member.islnk()` находят ссылки, `member.isfile()` — обычный файл.

Отклоняют абсолютные пути, `..`, ссылки и превышение общего лимита
распакованного размера. Если один участник опасен, не извлекают ничего.
`archive.extract(..., filter="data")` добавляет защиту, но не заменяет
собственную проверку политики архива. Параметр `filter` у `extract` появился
только в Python 3.12 (и в патч-версиях 3.9–3.11), так что на старом интерпретаторе
его может не быть — проверку путей приходится писать руками в любом случае.
У `add`, которым мы пользовались в разделе 7, параметр `filter` есть с Python 3.2
и работает везде.

### ZIP

ZIP одновременно упаковывает и обычно сжимает каждый файл отдельно, а в конце
файла держит оглавление. Поэтому из ZIP удобно достать одного участника, не
читая соседних. TAR сначала объединяет файлы в поток, а gzip/xz при
необходимости сжимает поток целиком — см. схему в разделе 7.

```python
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

report_path = Path("report.json")
report_path.write_text('{"status": "ok"}\n', encoding="utf-8")

with ZipFile("bundle.zip", "w", compression=ZIP_DEFLATED) as archive:
    archive.write(report_path, arcname="bundle/report.json")

with ZipFile("bundle.zip") as archive:
    print(archive.namelist())
    with archive.open("bundle/report.json") as file:
        content = file.read()
```

Для ZIP действуют те же правила доверия к путям участников: перед массовым
извлечением проверяют абсолютные пути и `..`.